# **Notebook 2: Data Preprocessing & Merging**

> **Objective:** Transform raw customer demographics and transactional basket data into a unified, model-ready master dataset. Heavy lifting and pipeline steps are modularly executed via our centralized `datacleaning.py` script.

---

## **Table of Contents**
1. [Imports and Libraries](#1-imports)
2. [Load Raw Datasets](#2-load)
3. [Execute Independent Cleaning Pipelines](#3-pipelines)
4. [Outlier Auditing & DBSCAN Validation](#4-outliers)
5. [The Master Merge (Inner Join)](#5-merge)
6. [Sanity and Validation Checks](#6-checks)
7. [Export Clean Dataset](#7-export)

---

## **1. Imports and Libraries <a id="1-imports"></a>**

In [ ]:
# to import from src folder (where our py files are located)
import sys 
sys.path.append("../src") 

# functions to clean the raw data
from datacleaning import customer_clean_data , basket_clean_data 

# to check the current working directory                      
import os 
os.getcwd()

import pandas as pd

# to automatically reload the py files when they are edited
%load_ext autoreload
%autoreload 2

# to ignore warnings
import warnings
warnings.filterwarnings('ignore')

## **2. Load Raw Datasets <a id="2-load"></a>**

> We import the 2 raw CSV files.

It is crucial to **understand the differences** between our two primary data sources before manipulating them:
- The `customer_info` dataset is a database where **each row represents a unique individual**. Therefore, we set the **customer ID** directly as the dataframe **index** to ensure data integrity.
- The `customer_basket` dataset is a transactional log of **receipts**. A single customer can have multiple rows (representing multiple trips to the store). Thus, we load it **without** a fixed index, which allows our pipeline to properly group and aggregate the transactions later.

In [ ]:
raw_data_customer = pd.read_csv('../data/customer_info.csv', index_col=0) #customer id as index
raw_data_basket = pd.read_csv('../data/customer_basket.csv')

print(f"Raw customer dataset shape: {raw_data_customer.shape}") 
print(f"Raw basket dataset shape: {raw_data_basket.shape}")

Raw customer dataset shape: (33038, 24)
Raw basket dataset shape: (100000, 3)


## **3. Execute Independent Cleaning Pipelines <a id = ' #3-pipelines'></a>**
> We run both datasets through their preprocessing **pipelines** pre-defined in our `datacleaning.py` file. These pipelines automatically correct data types, drop impossible values, handle missing values (using advanced KNN Imputation) and perform feature engineering (such as calculating customer's ages and another useful variables).

Our pipeline is divided into **two separate flows** that treat the specific needs of each dataset before they are merged:

#### **A. Customer Info (`customer_clean_data`)**

1. **Data Types:** We convert date strings into datetime objects, floats into integers and calculate `customer_age` based on the birthdate;
2. **Impossible Values:** We "created" some strict rules. Ages must be between 1 and 120, the `typical_hour` of shopping must be during store hours (6 to 23) and count/spend variables cannot be negative. Any violations are temporarily converted to `NaN`.
3. **Missing Values:** 
   * **Zero-filling:** Missing values in the `lifetime_spend` columns are filled with 0, assuming the customer simply hasn't bought from that category.
   * **KNN Imputation (k=7):** For the other missing values, we use a K-Nearest Neighbors imputer.
4. **Outliers:** We apply **DBSCAN** with `eps=3.5` and `min_samples=5` that automatically identifies and drops severe multidimensional outliers.
5. **Feature Engineering:** We create new analytical features:
   * `education_level`: Extracted directly from the customer's title (Bsc, Msc, Phd). Unknown when there's no information.
   * `total_children` & `has_children`: Aggregated from the kids and teens columns.
   * `time_of_day`: Categorized based on the typical hour.
   * `hour_sin` & `hour_cos`: Cyclical encoding of the shopping hour to preserve the continuous nature of time (e.g., 23:00 is close to 01:00).


#### **B. Customer Basket(`basket_clean_data`)**
1. **Data Parsing:** The `list_of_goods` comes in as a string. We use `ast.literal_eval` to convert these back into iterable Python objects.
2. **Feature Extraction:** We calculate the `basket_size` (number of items) for every single receipt.
3. **Aggregations:** Since the raw data has one row per receipt, we group the data by `customer_id` so it matches our Customer dataset. We aggregate the transactions to find:
   * `total_trips` (Number of unique invoices)
   * `total_items_bought` & `average_basket_size`
   * `unique_products_bought`
4. **Outlier Detection:** We run a second, tighter DBSCAN (`eps=1.5`, `min_samples=15`) specifically on the shopping behavior metrics to remove extreme bulk buyers.

--

In [5]:
# Run the complete cleaning pipeline
clean_customer_df = customer_clean_data(raw_data_customer)
clean_basket_df = basket_clean_data(raw_data_basket)

print(f"Cleaned customer dataset shape: {clean_customer_df.shape}")
print(f"Cleaned basket dataset shape: {clean_basket_df.shape}")

Cleaned customer dataset shape: (32269, 29)
Cleaned basket dataset shape: (28124, 7)


#### **Observed Changes**
As we can see from the outputs above, our pipeline successfully:
1. Shrank the customer dataset from **33,038** to **32,269** rows (mainly due to the removal of DBSCAN outliers).
2. Aggregated the **100,000** transactions into **28,124** unique customer profiles.

In the next step, we will run some validation checks to definitively prove that our pipeline resolved all missing values and successfully followed our logical constraints.

## **4. Merging the Two Clean Datasets (Inner Join)**

> In this section, we will combine the cleaned customer dataset with the aggregated transaction dataset using an **Inner Join**.

#### **Why Inner Join?**

We used this type of join to ensure that only customers with **complete information in both datasets** are included in the final analysis. We noticed many customers were only registred in one of them. This approach:

1. **Removes noise:** Eliminates customers with missing transaction history or incomplete profile data;
2. **Ensures consistency:** Only rows with matching IDs across both datasets are retained;
3. **Improves quality:** Prevents biased analyses caused by misaligned or incomplete data;
4. **Aligns with business logic:** We are only interested in customers with registered purchasing behavior.

In [8]:
final_df = pd.merge(
    clean_customer_df, 
    clean_basket_df.set_index('customer_id'), 
    left_index=True, 
    right_index=True, 
    how='inner'
)

final_df.to_csv('../data/final_dataset_clean.csv')

print(f"Final Dataset Final Shape: {final_df.shape}")
final_df.head()

Final Dataset Final Shape: (27475, 35)


,customer_name,customer_gender,customer_age,education_level,kids_home,teens_home,total_children,has_children,year_first_transaction,distinct_stores_visited,...,lifetime_spend_hygiene,lifetime_spend_petfood,latitude,longitude,total_trips,total_items_bought,average_basket_size,max_basket_size,min_basket_size,unique_products_bought
customer_id,,,,,,,,,,,,,,,,,,,,,
3,Crystal Kitchens,female,56,Bsc,1,1,2,1,2020,3,...,552.0,384.0,38.794428,-9.215739,2,21,10.5,11,10,21
4,Glenda Bauman,female,51,Bsc,1,0,1,1,2013,2,...,1880.0,665.0,38.751711,-9.179611,2,24,12.0,12,12,20
5,Antonio Campbell,male,55,Msc,0,0,0,0,2005,2,...,507.0,222.0,38.780678,-9.160656,1,8,8.0,8,8,8
7,John Kelling,male,44,Unknown,0,0,0,0,2021,1,...,485.0,184.0,38.739548,-9.148679,1,4,4.0,4,4,4
9,Nadine Garcia,female,58,Msc,1,1,2,1,2011,6,...,513.0,0.0,38.735577,-9.172423,2,20,10.0,14,6,20


We are left with a complete analysis of **27475 different customers**.

## **5. Validation Checks**

> We will now perform **validation checks** on our final dataset to verify **both pipelines were executed sucessfully.** Missing Values, Data types and our Logical Rules will be verified

#### **Check 1: Missing Values**

In [12]:
print('MISSING VALUES PER COLUMN:')
print('==========================')
missing_df = pd.DataFrame({
    'Column': final_df.columns,
    'Missing Count': final_df.isnull().sum().values,
    'Missing %': (final_df.isnull().sum().values / len(final_df) * 100).round(2)
})
display(missing_df.set_index('Column'))

MISSING VALUES PER COLUMN:


,Missing Count,Missing %
Column,,
customer_name,0,0.0
customer_gender,0,0.0
customer_age,0,0.0
education_level,0,0.0
kids_home,0,0.0
teens_home,0,0.0
total_children,0,0.0
has_children,0,0.0
year_first_transaction,0,0.0


We verify that the data cleaning pipeline has successfully resolved all missing values. A complete dataset 
with no null values is essential for machine learning models, as most algorithms cannot handle missing data 
without additional imputation strategies.

In [9]:
print('MISSING VALUES PER COLUMN:')
print('==========================')
print(final_df.isnull().sum())

MISSING VALUES PER COLUMN:
customer_name                              0
customer_gender                            0
customer_age                               0
education_level                            0
kids_home                                  0
teens_home                                 0
total_children                             0
has_children                               0
year_first_transaction                     0
distinct_stores_visited                    0
number_complaints                          0
typical_hour                               0
time_of_day                                0
hour_sin                                   0
hour_cos                                   0
lifetime_total_distinct_products           0
percentage_of_products_bought_promotion    0
lifetime_spend_groceries                   0
lifetime_spend_vegetables                  0
lifetime_spend_meat                        0
lifetime_spend_fish                        0
lifetime_spend_electronics  

We verify that the data cleaning pipeline has successfully resolved all missing values. A complete dataset 
with no null values is essential for machine learning models, as most algorithms cannot handle missing data 
without additional imputation strategies.

In [ ]:
# Verifies that columns are ordered nicely and types are correct (Int64, float64, etc.)
print('DATA TYPES PER COLUMN:')
print('======================')
print(clean_customer_df.dtypes)

DATA TYPES PER COLUMN:
customer_name                                   str
customer_gender                                 str
kids_home                                     Int64
teens_home                                    Int64
number_complaints                             Int64
distinct_stores_visited                       Int64
lifetime_spend_groceries                    float64
lifetime_spend_electronics                  float64
typical_hour                                  Int64
lifetime_spend_vegetables                   float64
lifetime_spend_nonalcohol_drinks            float64
lifetime_spend_alcohol_drinks               float64
lifetime_spend_meat                         float64
lifetime_spend_fish                         float64
lifetime_spend_hygiene                      float64
lifetime_spend_videogames                   float64
lifetime_spend_petfood                      float64
lifetime_total_distinct_products            float64
percentage_of_products_bought_promotion  

In [ ]:
# Describes min, max, and averages to prove no impossible values exist
print('SUMMARY STATISTICS FOR NUMERIC COLUMNS:')
print('=======================================')

columns_to_inspect = ['customer_age', 'typical_hour', 'percentage_of_products_bought_promotion', 'total_children']
print(clean_customer_df[columns_to_inspect].describe().loc[['min', 'max']])

SUMMARY STATISTICS FOR NUMERIC COLUMNS:
     customer_age  typical_hour  percentage_of_products_bought_promotion  \
min          24.0           6.0                                 0.000005   
max          86.0          23.0                                 1.000000   

     total_children  
min             0.0  
max            14.0  


<h2 style="color: #283593; border-bottom: 2px solid #5C6BC0; padding-bottom: 5px; margin-top: 30px;">5. Export Clean Data</h2>

In [ ]:
clean_customer_df.to_csv('../data/clean_customer_info.csv', index=True)

In [21]:
# 0. Importar as funções intermédias necessárias apenas para este teste
from datacleaning import eliminate_duplicates, basket_handle_datatypes, basket_feature_engineering, basket_aggregate

# 1. Isolar os IDs exatos dos clientes que o DBSCAN apagou
# Usando o nome correto da tua variável: raw_data_basket
todos_os_compradores = raw_data_basket['customer_id'].unique()
compradores_validos = clean_basket_df['customer_id'].unique()
outliers_ids = list(set(todos_os_compradores) - set(compradores_validos))

# 2. Recriar a tabela agregada ANTES de passar pelo filtro do DBSCAN
df_temp = eliminate_duplicates(raw_data_basket)
df_temp = basket_handle_datatypes(df_temp)
df_temp = basket_feature_engineering(df_temp)
df_agregado = basket_aggregate(df_temp)

# 3. Filtrar a tabela para mostrar APENAS os clientes expulsos
outliers_detetados = df_agregado[df_agregado['customer_id'].isin(outliers_ids)]

print(f"🔎 A inspecionar os {len(outliers_detetados)} clientes removidos pelo DBSCAN:")
display(outliers_detetados)

🔎 A inspecionar os 3 clientes removidos pelo DBSCAN:


,customer_id,total_trips,total_items_bought,average_basket_size,max_basket_size,min_basket_size,unique_products_bought
12151,17258,33,324,9.82,17,3,145
21059,29930,26,257,9.88,17,4,122
25237,35828,27,252,9.33,16,3,130
